# Bike Sharing Demand
## 从时间序列看城市出行规律 — 共享单车需求预测


In [ ]:
# === 基础库 ===
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# === 中文字体 ===
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# === 可视化设置 ===
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

print("基础库加载完成")

In [ ]:
# 加载数据
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')
print(f'训练集: {train.shape}')
print(f'测试集: {test.shape}')
train.head(3)

---

## 第 1 章 · 问题定义


### 1.1 任务描述

共享单车运营商需要提前知道"未来一小时某区域大概会租出多少辆车"，以便调配车辆、避免站点空置或堆积。Kaggle 提供了 Capital Bikeshare（华盛顿 DC）2011-2012 年的租车记录，我们用它来训练预测模型。

- **训练集**：10,886 条（每小时一条，每月前 19 天）
- **测试集**：6,493 条（每月 20 日到月底）
- **特征**：时间戳 + 季节/天气/温湿度/风速等 9 个字段
- **目标**：`count`（该小时的总租车数）


### 1.2 评估指标：RMSLE

Kaggle 用 **RMSLE**（Root Mean Squared Logarithmic Error）来评估：

$$\text{RMSLE} = \sqrt{\frac{1}{n} \sum_{i=1}^{n} \left(\log(y_i^{pred} + 1) - \log(y_i^{true} + 1)\right)^2}$$

跟常规的 RMSE 比，RMSLE 有两个特点：

1. **对大值和小值公平** — 取对数后，预测 500 辆时差 10 辆和预测 10 辆时差 1 辆的惩罚量级差不多。如果不取对数，模型会拼命拟合高峰时段而忽略深夜
2. **`+1` 的意义** — 避免 `log(0)` 炸掉（确实存在租车数为 0 的小时）

对建模的直接影响：**训练时对 `count` 做 `log1p` 变换，用 RMSE 作为损失函数，预测时 `expm1` 还原**。

In [ ]:
# RMSLE 评估函数
def rmsle(y_true, y_pred):
    return np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true)) ** 2))

def rmsle_score(model, X, y):
    """在 log1p 空间中做预测，计算 RMSLE"""
    y_pred_log = model.predict(X)
    y_pred = np.expm1(y_pred_log)
    return rmsle(y, y_pred)

print("RMSLE 评估函数已定义")

### 1.3 分析视角

一个城市的共享单车数据，本质上是用车记录了一座城市的作息规律。早高峰的骑行者是通勤族，下午两点骑行的是游客或自由职业者，深夜还在骑车的人有他们自己的故事。

接下来的分析不会只盯模型分数。我希望从数据里看到：**华盛顿 DC 的人什么时候出门、天气如何改变出行决策、通勤者和休闲骑行者的行为差异有多大**。

---

## 第 2 章 · 探索性数据分析（EDA）


In [ ]:
# 解析 datetime 字段（仅用于 EDA）
train['datetime'] = pd.to_datetime(train['datetime'])
test['datetime'] = pd.to_datetime(test['datetime'])

# 提取时间分量
train['hour'] = train['datetime'].dt.hour
train['day'] = train['datetime'].dt.day
train['weekday'] = train['datetime'].dt.dayofweek
train['month'] = train['datetime'].dt.month
train['year'] = train['datetime'].dt.year

print(f'时间范围: {train["datetime"].min()} ~ {train["datetime"].max()}')print(f'年份: {train["year"].unique()}')train[["datetime", "season", "holiday", "workingday", "weather"]].head(5)

### 2.1 数据总览


In [ ]:
# 基本信息
print("=== 数据类型 ===")
print(train.dtypes)
print(f"\n=== 缺失值 ===")
print(train.isnull().sum())
print(f"\n=== 描述统计 ===")
train.describe()

确认：**本数据集无缺失值**。`count` 的分布需要进一步分析。

---

### 2.2 目标变量分布


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 直方图
axes[0].hist(train["count"], bins=50, edgecolor="white", color="#2c7bb6")
axes[0].axvline(train["count"].mean(), color="red", linestyle="--",
               label=f'mean={train["count"].mean():.0f}')
axes[0].axvline(train["count"].median(), color="orange", linestyle="--",
               label=f'median={train["count"].median():.0f}')
axes[0].set_title("count distribution", fontsize=13)
axes[0].set_xlabel("count")
axes[0].legend()

# log1p 变换后
log_count = np.log1p(train["count"])
axes[1].hist(log_count, bins=50, edgecolor="white", color="#abd9e9")
axes[1].axvline(log_count.mean(), color="red", linestyle="--",
               label=f'mean={log_count.mean():.2f}')
axes[1].set_title("log1p(count) distribution", fontsize=13)
axes[1].set_xlabel("log1p(count)")
axes[1].legend()

# casual vs registered
axes[2].hist(train["casual"], bins=50, alpha=0.6, label="casual", color="#fdae61")
axes[2].hist(train["registered"], bins=50, alpha=0.6, label="registered", color="#2c7bb6")
axes[2].set_title("casual vs registered", fontsize=13)
axes[2].set_xlabel("count")
axes[2].legend()

plt.tight_layout()
plt.savefig("output/figures/count-distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print(f'count mean={train["count"].mean():.0f}, median={train["count"].median():.0f}, std={train["count"].std():.0f}')
print(f'casual share: {train["casual"].sum() / train["count"].sum() * 100:.1f}%')
print(f'registered share: {train["registered"].sum() / train["count"].sum() * 100:.1f}%')
print(f'hours with count=0: {(train["count"] == 0).sum()}')

`count` 呈典型的右偏分布 — 绝大多数时段租车量在 200 以下，极少数高峰超过 800。**log1p 变换后接近正态**，验证了用 log 空间建模的合理性。

registered 用户是主体（占约 82%），但 casual 有自己的行为模式，后面展开分析。

---

### 2.3 时间模式：小时 × 工作日

把一天 24 小时拉出来，按工作日/周末分开看——这是整份数据里信号最强的维度。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 工作日 vs 周末的每小时平均
wd = train[train["workingday"] == 1].groupby("hour")["count"].mean()
we = train[train["workingday"] == 0].groupby("hour")["count"].mean()

hours = range(24)
axes[0].plot(hours, wd, "o-", color="#2c7bb6", lw=2, ms=6, label="workingday")
axes[0].plot(hours, we, "s-", color="#fdae61", lw=2, ms=6, label="weekend/holiday")
axes[0].axvspan(7, 9, alpha=0.15, color="red")
axes[0].axvspan(17, 19, alpha=0.15, color="orange")
axes[0].set_xlabel("hour")
axes[0].set_ylabel("avg count")
axes[0].set_title("hourly avg: workingday vs weekend", fontsize=13)
axes[0].set_xticks(range(0, 24, 2))
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# casual vs registered 分时
ch = train.groupby("hour")["casual"].mean()
rh = train.groupby("hour")["registered"].mean()
axes[1].plot(hours, ch, "o-", color="#fdae61", lw=2, ms=6, label="casual")
axes[1].plot(hours, rh, "s-", color="#2c7bb6", lw=2, ms=6, label="registered")
axes[1].set_xlabel("hour")
axes[1].set_ylabel("avg count")
axes[1].set_title("hourly avg: casual vs registered", fontsize=13)
axes[1].set_xticks(range(0, 24, 2))
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("output/figures/hourly-patterns.png", dpi=150, bbox_inches="tight")
plt.show()

从小时模式能读出几个关键信息：

1. **工作日有两个尖峰** — 早 7-9 点和晚 17-19 点，典型的通勤潮汐。registered 用户是这两个尖峰的主力
2. **周末只有一个缓坡** — 从上午 10 点到下午 5 点维持高位，没有明显的早晚割裂
3. **casual 用户集中在白天** — 尤其是周末的下午，凌晨到清晨几乎为零
4. **深夜（0-5 点）租车量趋于 0** — 这个时段的预测难度最低

对特征工程的启示很明显：`hour` 是最重要的信号，但需要和 `workingday` 一起用。

---

### 2.4 季节与月份


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 月份
monthly = train.groupby("month")["count"].agg(["mean", "std"])
axes[0].bar(monthly.index, monthly["mean"], color="#2c7bb6", edgecolor="white")
axes[0].set_xlabel("month")
axes[0].set_ylabel("avg count")
axes[0].set_title("monthly avg", fontsize=13)
axes[0].set_xticks(range(1, 13))

# 季节
sl = {1: "spring", 2: "summer", 3: "fall", 4: "winter"}
sd = train.groupby("season")["count"].mean()
colors = ["#abd9e9", "#fdae61", "#d7191c", "#2c7bb6"]
axes[1].bar([sl[s] for s in sd.index], sd.values, color=colors, edgecolor="white")
axes[1].set_xlabel("season")
axes[1].set_ylabel("avg count")
axes[1].set_title("seasonal avg", fontsize=13)

# 年份
yl = train.groupby("year")["count"].mean()
axes[2].bar(yl.index.astype(str), yl.values, color=["#2c7bb6", "#abd9e9"], edgecolor="white")
axes[2].set_xlabel("year")
axes[2].set_ylabel("avg count")
axes[2].set_title("yearly avg", fontsize=13)

plt.tight_layout()
plt.savefig("output/figures/seasonal-patterns.png", dpi=150, bbox_inches="tight")
plt.show()
print(f'peak month: {monthly["mean"].idxmax():.0f} (avg {monthly["mean"].max():.0f})')
print(f'lowest month: {monthly["mean"].idxmin():.0f} (avg {monthly["mean"].min():.0f})')
print(f'2011->2012 growth: {(yl[2012] / yl[2011] - 1) * 100:.0f}%')

季节和年份的趋势很清晰：夏天和秋天是骑行旺季，冬天最低。2012 年比 2011 年整体增长了约 70%——共享单车在快速普及。

测试集的日期覆盖了 `train` 没有的时段（每月 20 日之后），所以年份趋势和月内日期位置都值得作为特征。

---

### 2.5 天气的影响


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# weather
wl = {1: "clear", 2: "cloudy", 3: "light rain/snow", 4: "heavy rain/snow"}
wd2 = train.groupby("weather")["count"].agg(["mean", "count"])
axes[0, 0].bar([wl.get(w, str(w)) for w in wd2.index], wd2["mean"], color="#2c7bb6", edgecolor="white")
axes[0, 0].set_title("count by weather", fontsize=13)
axes[0, 0].set_ylabel("avg count")
for i, (m, c) in enumerate(zip(wd2["mean"], wd2["count"])):
    axes[0, 0].text(i, m + 5, f"n={c}", ha="center", fontsize=9)

# temp vs count
axes[0, 1].scatter(train["temp"], train["count"], alpha=0.3, s=5, color="#2c7bb6")
axes[0, 1].set_xlabel("temp (C)")
axes[0, 1].set_ylabel("count")
axes[0, 1].set_title("temp vs count", fontsize=13)

# humidity vs count
axes[0, 2].scatter(train["humidity"], train["count"], alpha=0.3, s=5, color="#fdae61")
axes[0, 2].set_xlabel("humidity")
axes[0, 2].set_ylabel("count")
axes[0, 2].set_title("humidity vs count", fontsize=13)

# temp vs atemp
axes[1, 0].scatter(train["temp"], train["atemp"], alpha=0.3, s=5, color="#d7191c")
axes[1, 0].plot([0, 45], [0, 45], "k--", alpha=0.3)
axes[1, 0].set_xlabel("temp")
axes[1, 0].set_ylabel("atemp")
axes[1, 0].set_title("temp vs atemp", fontsize=13)
r = train["temp"].corr(train["atemp"])
axes[1, 0].text(5, 40, f"r={r:.4f}", fontsize=11)

# windspeed vs count
axes[1, 1].scatter(train["windspeed"], train["count"], alpha=0.3, s=5, color="#abd9e9")
axes[1, 1].set_xlabel("windspeed")
axes[1, 1].set_ylabel("count")
axes[1, 1].set_title("windspeed vs count", fontsize=13)

# 风速=0
zw = (train["windspeed"] == 0).sum()
axes[1, 2].hist(train["windspeed"], bins=40, edgecolor="white", color="#2c7bb6")
axes[1, 2].axvline(0, color="red", linestyle="--")
axes[1, 2].set_xlabel("windspeed")
axes[1, 2].set_title(f"windspeed dist (0: {zw}, {zw/len(train)*100:.1f}%)", fontsize=13)

plt.tight_layout()
plt.savefig("output/figures/weather-analysis.png", dpi=150, bbox_inches="tight")
plt.show()

天气角度的发现：

1. **极端天气（weather=4）只有 1 条记录** — 模型基本学不到这个类别的信号
2. **温度和租车量正相关，但有天花板效应** — 太热（>35°C）后租车量反而下降
3. **temp 和 atemp 几乎完全共线（r≈0.99）** — 建模时保留一个即可
4. **高湿度时段租车量偏低**，但关系较弱
5. **风速为零的比例约 13%** — 传感器故障的可能性很大，需要在清洗阶段处理

---

### 2.6 casual vs registered 的行为差异

这是决定后续建模策略（分头预测）的关键依据。

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 9))
w = 0.35

# 工作日 vs 周末
wc = train.groupby("workingday")["casual"].mean()
wr = train.groupby("workingday")["registered"].mean()
x = np.array([0, 1])
axes[0, 0].bar(x - w/2, wc, w, color="#fdae61", label="casual")
axes[0, 0].bar(x + w/2, wr, w, color="#2c7bb6", label="registered")
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels(["weekend/holiday", "workingday"])
axes[0, 0].set_ylabel("avg count")
axes[0, 0].set_title("workingday vs weekend", fontsize=13)
axes[0, 0].legend()

# 天气
wdc = train.groupby("weather")["casual"].mean()
wdr = train.groupby("weather")["registered"].mean()
x_w = np.arange(len(wdc))
axes[0, 1].bar(x_w - w/2, wdc, w, color="#fdae61", label="casual")
axes[0, 1].bar(x_w + w/2, wdr, w, color="#2c7bb6", label="registered")
axes[0, 1].set_xticks(x_w)
axes[0, 1].set_xticklabels([wl[w] for w in wdc.index], fontsize=9)
axes[0, 1].set_ylabel("avg count")
axes[0, 1].set_title("different weather", fontsize=13)
axes[0, 1].legend()

# 季节
sc = train.groupby("season")["casual"].mean()
sr = train.groupby("season")["registered"].mean()
x_s = np.arange(len(sc))
axes[1, 0].bar(x_s - w/2, sc, w, color="#fdae61", label="casual")
axes[1, 0].bar(x_s + w/2, sr, w, color="#2c7bb6", label="registered")
axes[1, 0].set_xticks(x_s)
axes[1, 0].set_xticklabels([sl[s] for s in sc.index])
axes[1, 0].set_ylabel("avg count")
axes[1, 0].set_title("different seasons", fontsize=13)
axes[1, 0].legend()

# 每小时 casual 占比
cr = train.groupby("hour")["casual"].sum() / train.groupby("hour")["count"].sum()
axes[1, 1].bar(hours, cr.values * 100, color="#fdae61", edgecolor="white")
axes[1, 1].axhline(y=cr.mean() * 100, color="red", linestyle="--", alpha=0.5, label=f"overall {cr.mean()*100:.0f}%")
axes[1, 1].set_xlabel("hour")
axes[1, 1].set_ylabel("casual ratio (%)")
axes[1, 1].set_title("casual ratio by hour", fontsize=13)
axes[1, 1].set_xticks(range(0, 24, 2))
axes[1, 1].legend()

plt.tight_layout()
plt.savefig("output/figures/casual-vs-registered.png", dpi=150, bbox_inches="tight")
plt.show()

casual 和 registered 的用户画像完全不同：

| 维度 | casual | registered |
|------|--------|------------|
| 工作日 vs 周末 | 周末高 | 工作日高 |
| 小时分布 | 中午到傍晚 | 早晚双峰 |
| 天气敏感度 | 极高（坏天骤降） | 中（通勤刚需） |
| 季节偏好 | 夏秋高，冬春低 | 全年相对平稳 |

**结论：两者由不同的因素驱动，分头建模有理论支撑。**

---

### 2.7 EDA 小结

通过这几轮观察，关键发现整理如下：

1. **hour 是最强信号** — 不同小时的租车量差异可达 10 倍以上
2. **hour 的效果取决于 workingday** — 同样的早 8 点，工作日 vs 周末完全不同
3. **temp 和 atemp 只保留一个** — 共线性太高（r≈0.99）
4. **windspeed=0 不自然** — 高达 13%，需要处理
5. **casual 和 registered 行为模式分化明显** — 分头建模是合理选择
6. **2012 年比 2011 年增长了约 70%** — year 是必要的特征

接下来：构造特征、处理异常值、建立 baseline。

---

## 第 3 章 · 特征工程


### 3.1 特征工程策略

基于 EDA 的发现，设计以下特征：

| 类别 | 特征 | 构造方式 |
|------|------|----------|
| 时间分解 | `hour`, `weekday`, `month`, `year` | 从 datetime 直接提取 |
| 时段标记 | `day_period` | 凌晨(0-5) / 上午(6-11) / 下午(12-17) / 晚上(18-23) |
| 高峰标记 | `is_rush_morning`, `is_rush_evening` | 工作日 7-9 / 17-19 |
| 周末标记 | `is_weekend` | weekday >= 5 |
| 交互特征 | `hour×workingday`, `hour×season` | 捕获时间维度的条件效应 |
| 天气交互 | `temp×humidity` | 体感舒适度代理 |

**不引入滞后特征**：测试集没有历史 count，迭代预测会放大误差，收益/成本比不划算。

In [ ]:
# === 合并数据，统一构造特征 ===
all_data = pd.concat([train.drop(['casual', 'registered', 'count'], axis=1), test],
                     axis=0, sort=False, ignore_index=True)

# === 时间分解 ===
all_data['hour'] = all_data['datetime'].dt.hour
all_data['weekday'] = all_data['datetime'].dt.dayofweek  # 0=Mon, 6=Sun
all_data['month'] = all_data['datetime'].dt.month
all_data['year'] = all_data['datetime'].dt.year
all_data['day'] = all_data['datetime'].dt.day

# === 周末标记 ===
all_data['is_weekend'] = (all_data['weekday'] >= 5).astype(int)

# === 时段分类 ===
def classify_period(h):
    if h <= 5: return 0   # 凌晨
    elif h <= 11: return 1 # 上午
    elif h <= 17: return 2 # 下午
    else: return 3          # 晚上
all_data['day_period'] = all_data['hour'].apply(classify_period)

# === 高峰时段标记 ===
all_data['is_rush_morning'] = ((all_data['hour'] >= 7) & (all_data['hour'] <= 9) &
                                 (all_data['workingday'] == 1)).astype(int)
all_data['is_rush_evening'] = ((all_data['hour'] >= 17) & (all_data['hour'] <= 19) &
                                (all_data['workingday'] == 1)).astype(int)

# === 交互特征 ===
all_data['hour_x_workingday'] = all_data['hour'] * all_data['workingday']
all_data['hour_x_season'] = all_data['hour'] * all_data['season']
all_data['temp_x_humidity'] = all_data['temp'] * all_data['humidity']

# === 验证 ===
print(f'特征维度: {all_data.shape[1]}')
all_data.head(3)

### 3.2 特征选择

把构造好的特征整理成最终的特征列表。排除原始 datetime 和冗余字段。

In [ ]:
# === 最终特征列表 ===
features = [
    'season', 'holiday', 'workingday', 'weather',
    'temp', 'humidity', 'windspeed',          # 天气（去掉 atemp，与 temp 共线 r≈0.99）
    'hour', 'weekday', 'month', 'year', 'day', # 时间分解
    'is_weekend', 'day_period',                 # 时段标记
    'is_rush_morning', 'is_rush_evening',       # 高峰标记
    'hour_x_workingday', 'hour_x_season',       # 时间交互
    'temp_x_humidity',                          # 天气交互
]

print(f'特征数量: {len(features)}')
print(features)

---

## 第 4 章 · 数据清洗


### 4.1 windspeed=0 处理

13% 的风速数据为 0，物理上不太可能出现 —— DC 不是全年无风的地方。更可能是传感器故障。

处理策略：按天气条件分组，用各组中位数填充风速为 0 的记录。这样比全局中位数更精细。

In [ ]:
# === 处理 windspeed=0 ===
zero_wind_mask = all_data['windspeed'] == 0
print(f'windspeed=0 的记录: {zero_wind_mask.sum()} ({zero_wind_mask.sum()/len(all_data)*100:.1f}%)')

# 按 weather 条件分组填充中位数
for weather_type in all_data['weather'].unique():
    mask = (all_data['weather'] == weather_type) & (all_data['windspeed'] == 0)
    median_wind = all_data.loc[(all_data['weather'] == weather_type) & (all_data['windspeed'] > 0), 'windspeed'].median()
    if pd.notna(median_wind):
        all_data.loc[mask, 'windspeed'] = median_wind
        count = mask.sum()
        if count > 0:
            print(f'  weather={weather_type}: 填充 {count} 条 → 中位数 {median_wind:.2f}')

print(f'处理后 windspeed=0: {(all_data["windspeed"] == 0).sum()} 条')

### 4.2 humidity 异常值

湿度理论范围是 0-100。检查是否有超出范围或明显不合理的数据。

In [ ]:
# === humidity 检查 ===
print(f'humidity 范围: {all_data["humidity"].min():.0f} ~ {all_data["humidity"].max():.0f}')
print(f'humidity=0: {(all_data["humidity"] == 0).sum()} 条')
print(f'humidity=100: {(all_data["humidity"] == 100).sum()} 条')
print(f'humidity < 10: {(all_data["humidity"] < 10).sum()} 条')

# humidity 为 0 的极端情况 → 用 weather 分组中位数填充
zero_humid = all_data['humidity'] == 0
print(f'\nhumidity=0 的 weather 分布:')
print(all_data.loc[zero_humid, "weather"].value_counts())

In [ ]:
# === humidity=0 填充 ===
for weather_type in all_data['weather'].unique():
    mask = (all_data['weather'] == weather_type) & (all_data['humidity'] == 0)
    median_h = all_data.loc[(all_data['weather'] == weather_type) & (all_data['humidity'] > 0), 'humidity'].median()
    if pd.notna(median_h):
        all_data.loc[mask, 'humidity'] = median_h

print(f'处理后 humidity=0: {(all_data["humidity"] == 0).sum()} 条')

### 4.3 temp 和 atemp 共线性

EDA 阶段确认了 temp 和 atemp 的相关系数 ≈ 0.99。保留 `temp`，去掉 `atemp`（已在特征列表中体现）。

### 4.4 数据分割

特征构造和清洗完成后，重新切分训练集和测试集。

In [ ]:
# === 重新切分 ===
train_len = len(train)
train_processed = all_data[:train_len].copy()
test_processed = all_data[train_len:].copy()

# 目标变量（log1p 变换）
y_count = np.log1p(train['count'].values)
y_casual = np.log1p(train['casual'].values)
y_registered = np.log1p(train['registered'].values)

# 提取特征矩阵
X = train_processed[features].values
X_test = test_processed[features].values

print(f'X: {X.shape}, y_count: {y_count.shape}')
print(f'X_test: {X_test.shape}')
print(f'特征完整性: null={np.isnan(X).sum()}, inf={np.isinf(X).sum()}')

### 4.5 清洗小结

| 问题 | 处理方式 |
|------|----------|
| windspeed=0（13%） | 按 weather 分组中位数填充 |
| humidity=0 | 按 weather 分组中位数填充 |
| temp/atemp 共线（r≈0.99） | 保留 temp，移除 atemp |

接下来进入建模阶段。

---

## 第 5 章 · Baseline 建模


### 5.1 建模准备

回顾目前的准备状态：
- 特征矩阵 X (17 维) 已构造完成
- y_count / y_casual / y_registered 已 log1p 变换
- 训练集 / 测试集已划分

评估统一用 **RMSLE**，交叉验证用 **TimeSeriesSplit**（按时间顺序划分，不用随机打乱）。

In [ ]:
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import make_scorer

# === RMSLE scorer（用于 cross_val_score） ===
def rmsle_metric(y_true_log, y_pred_log):
    """输入已是 log1p 空间，先还原再算 RMSLE"""
    y_true = np.expm1(y_true_log)
    y_pred = np.expm1(y_pred_log)
    return np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true)) ** 2))

rmsle_scorer = make_scorer(rmsle_metric, greater_is_better=False)

# === 时间序列 CV ===
tscv = TimeSeriesSplit(n_splits=5)
print("TimeSeriesSplit(5) 已就绪")
print(f"训练集大小: {len(X)}, 测试集大小: {len(X_test)}")

### 5.2 Baseline 模型（直接预测 count）

先用 4 个基础模型直接预测 `log1p(count)`，不做调参，仅看 baseline 水平。

In [ ]:
# === 定义基础模型（默认参数） ===
base_models = {
    'Ridge': Ridge(alpha=1.0, random_state=42),
    'RandomForest': RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1),
    'XGBoost': XGBRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42, verbosity=0),
    'LightGBM': LGBMRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42, verbose=-1),
}

# === 评估每个模型（预测 count） ===
print('=== Baseline: 直接预测 count ===\n')
results_count = {}
for name, model in base_models.items():
    scores = []
    for fold, (train_idx, val_idx) in enumerate(tscv.split(X)):
        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr, y_val = y_count[train_idx], y_count[val_idx]
        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_val)
        score = rmsle_metric(y_val, y_pred)
        scores.append(score)
        print(f'  {name} Fold{fold+1}: RMSLE={score:.4f}')
    
    results_count[name] = {"mean": np.mean(scores), "std": np.std(scores)}
    print(f'  → {name} Avg RMSLE: {np.mean(scores):.4f} (+/- {np.std(scores):.4f})\n')

# === 汇总 ===
print('=== Direct predict summary ===')
for name, r in sorted(results_count.items(), key=lambda x: x[1]["mean"]):
    print(f'  {name:15s}  RMSLE={r["mean"]:.4f} (+/- {r["std"]:.4f})')

### 5.3 分头预测策略

EDA 阶段发现 casual 和 registered 的行为模式差异很大。尝试分别建模：

1. Model_A 预测 `log1p(casual)`
2. Model_B 预测 `log1p(registered)`
3. 最终预测 = `expm1(pred_casual) + expm1(pred_registered)`

这个策略的理论基础：两类用户几乎不受相同的因素驱动，分开建模让每个模型专注学自己该学的东西。

In [ ]:
# === 分头预测：每个模型分别预测 casual 和 registered ===
print('=== Split predict: casual + registered ===\n')
results_split = {}

for name, model_cls in [
    ('Ridge', Ridge(alpha=1.0, random_state=42)),
    ('RandomForest', RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)),
    ('XGBoost', XGBRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42, verbosity=0)),
    ('LightGBM', LGBMRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42, verbose=-1)),
]:
    scores = []
    for fold, (train_idx, val_idx) in enumerate(tscv.split(X)):
        X_tr, X_val = X[train_idx], X[val_idx]
        
        # 预测 casual
        m_c = model_cls.__class__(**model_cls.get_params())
        m_c.fit(X_tr, y_casual[train_idx])
        pred_casual_log = m_c.predict(X_val)
        
        # 预测 registered
        m_r = model_cls.__class__(**model_cls.get_params())
        m_r.fit(X_tr, y_registered[train_idx])
        pred_registered_log = m_r.predict(X_val)
        
        # 合并
        pred_total = np.expm1(pred_casual_log) + np.expm1(pred_registered_log)
        y_true_total = np.expm1(y_count[val_idx])
        score = rmsle(y_true_total, pred_total)
        scores.append(score)
        print(f'  {name} Fold{fold+1}: RMSLE={score:.4f}')
    
    results_split[name] = {"mean": np.mean(scores), "std": np.std(scores)}
    print(f'  → {name} Avg RMSLE: {np.mean(scores):.4f} (+/- {np.std(scores):.4f})\n')

# === 对比 ===
print('=== Direct vs Split comparison ===')
print(f'{"Model":15s}  {"Direct":>10s}  {"Split":>10s}  {"Improvement":>12s}')
print('-' * 52)
for name in results_count:
    d = results_count[name]["mean"]
    s = results_split[name]["mean"]
    imp = (d - s) / d * 100
    print(f'{name:15s}  {d:10.4f}  {s:10.4f}  {imp:>+11.1f}%')

### 5.4 Baseline 总结

对比实验的结论决定了后续方向：

1. **最佳单一模型是哪个？** — 决定了调参的主力
2. **分头预测是否优于直接预测？** — 如果是，后续调参和集成统一用 split 策略
3. **树模型（XGB/LGB）vs 线性模型（Ridge）** — 决定了集成的互补性

接下来：对表现最好的模型做深度调参。

---

## 第 6 章 · 深度调参


### 6.1 调参策略

用 `RandomizedSearchCV` 替代 GridSearchCV：参数量大时，随机搜索能更快找到近似最优。

- **CV**：TimeSeriesSplit(5)，跟 baseline 一致
- **Scoring**：RMSLE（neg RMSLE for sklearn）
- **搜索次数**：每个模型 50 组随机参数
- **重点模型**：XGBoost 和 LightGBM（梯度提升树通常最优），RandomForest 做轻量搜索

所有调参都在 **分头预测** 框架下进行 — 即分别对 casual 和 registered 做搜索。

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

print("RandomizedSearchCV 已导入")

### 6.2 XGBoost 调参


In [ ]:
# === XGBoost 超参空间 ===
xgb_params = {
    'n_estimators': randint(100, 500),
    'max_depth': randint(3, 10),
    'learning_rate': uniform(0.01, 0.2),
    'subsample': uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4),
    'reg_alpha': uniform(0, 2),
    'reg_lambda': uniform(0.5, 5),
    'min_child_weight': randint(1, 10),
}

xgb_search_casual = RandomizedSearchCV(
    XGBRegressor(random_state=42, verbosity=0),
    xgb_params, n_iter=50, cv=tscv,
    scoring='neg_root_mean_squared_error',
    random_state=42, n_jobs=-1, verbose=0
)
xgb_search_casual.fit(X, y_casual)
print(f'XGB casual best: RMSLE={-xgb_search_casual.best_score_:.4f}')
print(f'  params: {xgb_search_casual.best_params_}')

xgb_search_registered = RandomizedSearchCV(
    XGBRegressor(random_state=42, verbosity=0),
    xgb_params, n_iter=50, cv=tscv,
    scoring='neg_root_mean_squared_error',
    random_state=42, n_jobs=-1, verbose=0
)
xgb_search_registered.fit(X, y_registered)
print(f'\nXGB registered best: RMSLE={-xgb_search_registered.best_score_:.4f}')
print(f'  params: {xgb_search_registered.best_params_}')

### 6.3 LightGBM 调参


In [ ]:
# === LightGBM 超参空间 ===
lgb_params = {
    'n_estimators': randint(100, 500),
    'max_depth': randint(3, 12),
    'num_leaves': randint(20, 100),
    'learning_rate': uniform(0.01, 0.2),
    'subsample': uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4),
    'reg_alpha': uniform(0, 2),
    'reg_lambda': uniform(0.5, 5),
    'min_child_samples': randint(10, 50),
}

lgb_search_casual = RandomizedSearchCV(
    LGBMRegressor(random_state=42, verbose=-1),
    lgb_params, n_iter=50, cv=tscv,
    scoring='neg_root_mean_squared_error',
    random_state=42, n_jobs=-1, verbose=0
)
lgb_search_casual.fit(X, y_casual)
print(f'LGB casual best: RMSLE={-lgb_search_casual.best_score_:.4f}')
print(f'  params: {lgb_search_casual.best_params_}')

lgb_search_registered = RandomizedSearchCV(
    LGBMRegressor(random_state=42, verbose=-1),
    lgb_params, n_iter=50, cv=tscv,
    scoring='neg_root_mean_squared_error',
    random_state=42, n_jobs=-1, verbose=0
)
lgb_search_registered.fit(X, y_registered)
print(f'\nLGB registered best: RMSLE={-lgb_search_registered.best_score_:.4f}')
print(f'  params: {lgb_search_registered.best_params_}')

### 6.4 RandomForest 调参（轻量）


In [ ]:
# === RF 超参空间（轻量搜索 30 组） ===
rf_params = {
    'n_estimators': randint(100, 400),
    'max_depth': randint(5, 20),
    'min_samples_split': randint(2, 15),
    'min_samples_leaf': randint(1, 10),
    'max_features': ['sqrt', 'log2', None],
}

rf_search_casual = RandomizedSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    rf_params, n_iter=30, cv=tscv,
    scoring='neg_root_mean_squared_error',
    random_state=42, n_jobs=-1, verbose=0
)
rf_search_casual.fit(X, y_casual)
print(f'RF casual best: RMSLE={-rf_search_casual.best_score_:.4f}')
print(f'  params: {rf_search_casual.best_params_}')

rf_search_registered = RandomizedSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    rf_params, n_iter=30, cv=tscv,
    scoring='neg_root_mean_squared_error',
    random_state=42, n_jobs=-1, verbose=0
)
rf_search_registered.fit(X, y_registered)
print(f'\nRF registered best: RMSLE={-rf_search_registered.best_score_:.4f}')
print(f'  params: {rf_search_registered.best_params_}')

### 6.5 调参总结

对比调参前后，确认调参的收益。同时记录最优参数，为集成做准备。

In [ ]:
# === 调参前后对比（分头预测框架下） ===
print('=== Tuning Summary (split-predict framework) ===\n')

# 构建调参后的最优模型
best_models = {
    'XGBoost': (
        xgb_search_casual.best_estimator_,
        xgb_search_registered.best_estimator_
    ),
    'LightGBM': (
        lgb_search_casual.best_estimator_,
        lgb_search_registered.best_estimator_
    ),
    'RandomForest': (
        rf_search_casual.best_estimator_,
        rf_search_registered.best_estimator_
    ),
}

print(f'{"Model":15s} {"Baseline":>10s} {"Tuned":>10s} {"Improvement":>12s}')
print('-' * 52)
baseline_ref = {
    'XGBoost': results_split.get('XGBoost', {}).get('mean', 0),
    'LightGBM': results_split.get('LightGBM', {}).get('mean', 0),
    'RandomForest': results_split.get('RandomForest', {}).get('mean', 0),
}

for name in best_models:
    # 在完整训练集上评估调参后模型
    m_c, m_r = best_models[name]
    tuned_scores = []
    for fold, (train_idx, val_idx) in enumerate(tscv.split(X)):
        X_tr, X_val = X[train_idx], X[val_idx]
        m_c.fit(X_tr, y_casual[train_idx])
        m_r.fit(X_tr, y_registered[train_idx])
        pred_total = np.expm1(m_c.predict(X_val)) + np.expm1(m_r.predict(X_val))
        score = rmsle(np.expm1(y_count[val_idx]), pred_total)
        tuned_scores.append(score)
    tuned_mean = np.mean(tuned_scores)
    base = baseline_ref.get(name, 0)
    imp = (base - tuned_mean) / base * 100 if base > 0 else 0
    print(f'{name:15s}  {base:10.4f}  {tuned_mean:10.4f}  {imp:>+11.1f}%')

---

## 第 7 章 · 集成与 Stacking


### 7.1 集成策略

用 `StackingRegressor` 将调参后的最优模型做两层集成：

- **Base estimators**：XGBoost + LightGBM + RandomForest（调参后最优）
- **Meta learner**：Ridge Regression
- **CV**：TimeSeriesSplit(5)，保持一致性

对 casual 和 registered 各建一个 Stacking，然后合并预测。

In [ ]:
from sklearn.ensemble import StackingRegressor

# === 为 casual 构建 Stacking ===
base_estimators_casual = [
    ('xgb', xgb_search_casual.best_estimator_),
    ('lgb', lgb_search_casual.best_estimator_),
    ('rf', rf_search_casual.best_estimator_),
]

stack_casual = StackingRegressor(
    estimators=base_estimators_casual,
    final_estimator=Ridge(alpha=1.0, random_state=42),
    cv=tscv, n_jobs=-1
)

# === 为 registered 构建 Stacking ===
base_estimators_registered = [
    ('xgb', xgb_search_registered.best_estimator_),
    ('lgb', lgb_search_registered.best_estimator_),
    ('rf', rf_search_registered.best_estimator_),
]

stack_registered = StackingRegressor(
    estimators=base_estimators_registered,
    final_estimator=Ridge(alpha=1.0, random_state=42),
    cv=tscv, n_jobs=-1
)

print("StackingRegressor 已构建（casual + registered）")

### 7.2 最终对比实验

三种策略放在一起比，确认最优方案：

1. **Direct** — 最优单一模型直接预测 count
2. **Split** — 最优单一模型分头预测 casual + registered
3. **Stacking** — 分头预测 + Stacking 集成

用 TimeSeriesSplit 跑完 5-fold，比较 RMSLE 均值。

In [ ]:
# === 策略 3: Stacking 分头预测 ===
print('=== Strategy 3: Stacking + Split predict ===\n')
stack_scores = []
for fold, (train_idx, val_idx) in enumerate(tscv.split(X)):
    X_tr, X_val = X[train_idx], X[val_idx]
    
    # casual stacking
    stack_casual.fit(X_tr, y_casual[train_idx])
    pred_casual_log = stack_casual.predict(X_val)
    
    # registered stacking
    stack_registered.fit(X_tr, y_registered[train_idx])
    pred_registered_log = stack_registered.predict(X_val)
    
    # 合并
    pred_total = np.expm1(pred_casual_log) + np.expm1(pred_registered_log)
    y_true_total = np.expm1(y_count[val_idx])
    score = rmsle(y_true_total, pred_total)
    stack_scores.append(score)
    print(f'  Fold{fold+1}: RMSLE={score:.4f}')

print(f'\nStacking Avg RMSLE: {np.mean(stack_scores):.4f} (+/- {np.std(stack_scores):.4f})')

In [ ]:
# === 最终对比汇总 ===
print('=== Final Comparison ===')
print(f'{"Strategy":25s} {"RMSLE":>10s}')
print('-' * 40)

# Direct（取 baseline 里最佳的 direct 模型）
best_direct = min(results_count.values(), key=lambda x: x["mean"])
print(f'{"1. Direct (best single)":25s}  {best_direct["mean"]:10.4f}')

# Split（取 baseline 里最佳的 split 模型）
best_split = min(results_split.values(), key=lambda x: x["mean"])
print(f'{"2. Split (best single)":25s}  {best_split["mean"]:10.4f}')

# Stacking
print(f'{"3. Stacking (split)":25s}  {np.mean(stack_scores):10.4f}')

print('\n=== 结论 ===')
all_means = [best_direct["mean"], best_split["mean"], np.mean(stack_scores)]
best_strategy = ['Direct', 'Split', 'Stacking'][np.argmin(all_means)]
print(f'最优策略: {best_strategy} (RMSLE={min(all_means):.4f})')
print(f'vs Direct 提升: {(best_direct["mean"] - min(all_means)) / best_direct["mean"] * 100:.1f}%')

### 7.3 生成最终提交


In [ ]:
# === 用最优策略在全量训练集上训练，生成预测 ===
print("在全量训练集上训练最终模型...")

# 全量训练 stacking
stack_casual.fit(X, y_casual)
stack_registered.fit(X, y_registered)

# 预测测试集
test_pred_casual_log = stack_casual.predict(X_test)
test_pred_registered_log = stack_registered.predict(X_test)

test_pred = np.expm1(test_pred_casual_log) + np.expm1(test_pred_registered_log)
test_pred = np.maximum(test_pred, 0)  # 确保非负

# 生成提交文件
submission = pd.DataFrame({
    'datetime': test['datetime'],
    'count': test_pred
})
submission.to_csv('output/submission.csv', index=False)

print(f'预测范围: {test_pred.min():.0f} ~ {test_pred.max():.0f}')
print(f'预测均值: {test_pred.mean():.0f}')
print(f'预测中位数: {np.median(test_pred):.0f}')
print(f'提交文件: output/submission.csv ({len(submission)} 条)')

---

## 第 8 章 · 复盘与总结


### 8.1 最优模型与策略

完整回顾整个建模过程采用的策略和最终选择。

In [ ]:
# === 最终模型总结 ===
print('=== Final Model Summary ===')
print(f'特征数量: {len(features)}')
print(f'CV 策略: TimeSeriesSplit(5)')
print(f'目标变换: log1p')
print(f'集成策略: Stacking (XGB + LGB + RF) → Ridge meta-learner')
print(f'预测策略: 分头预测 casual + registered')

# 各模型的最优参数
print('\n=== XGBoost (casual) 最优参数 ===')
print(xgb_search_casual.best_params_)
print('\n=== LightGBM (registered) 最优参数 ===')
print(lgb_search_registered.best_params_)

### 8.2 特征重要性

从 XGBoost 模型中提取特征重要性。由于是分头预测，分别看 casual 和 registered 的重要特征排序。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# casual 模型的特征重要性
imp_casual = xgb_search_casual.best_estimator_.feature_importances_
indices_c = np.argsort(imp_casual)[-10:]
axes[0].barh(range(10), imp_casual[indices_c], color="#fdae61", edgecolor="white")
axes[0].set_yticks(range(10))
axes[0].set_yticklabels([features[i] for i in indices_c])
axes[0].set_xlabel("importance")
axes[0].set_title("top 10 features: casual model", fontsize=13)

# registered 模型的特征重要性
imp_registered = xgb_search_registered.best_estimator_.feature_importances_
indices_r = np.argsort(imp_registered)[-10:]
axes[1].barh(range(10), imp_registered[indices_r], color="#2c7bb6", edgecolor="white")
axes[1].set_yticks(range(10))
axes[1].set_yticklabels([features[i] for i in indices_r])
axes[1].set_xlabel("importance")
axes[1].set_title("top 10 features: registered model", fontsize=13)

plt.tight_layout()
plt.savefig("output/figures/feature-importance.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# === 对比分析：casual vs registered 特征差异 ===
print('=== casual top 5 ===')
for i in indices_c[::-1][:5]:
    print(f'  {features[i]:20s}  {imp_casual[i]:.4f}')

print('\n=== registered top 5 ===')
for i in indices_r[::-1][:5]:
    print(f'  {features[i]:20s}  {imp_registered[i]:.4f}')

### 8.3 SHAP 分析

用 SHAP 理解关键特征如何影响预测。取 casual 模型的一个子集做分析（全量太慢）。

In [ ]:
import shap

# === 采样分析 ===
X_sample = X[np.random.choice(len(X), 500, replace=False)]
explainer = shap.TreeExplainer(xgb_search_casual.best_estimator_)
shap_values = explainer.shap_values(X_sample)

# SHAP summary（casual 模型）
fig, ax = plt.subplots(figsize=(12, 7))
shap.summary_plot(shap_values, X_sample, feature_names=features, show=False)
plt.title("SHAP summary: casual model", fontsize=14, fontweight="normal")
plt.savefig("output/figures/shap-casual.png", dpi=150, bbox_inches="tight")
plt.show()

### 8.4 城市出行洞察

回到最初的分析视角——从数据里看一座城市的作息规律。

#### 通勤 vs 休闲：两种骑行节奏

registered 用户（通勤族）的行为像一个定时闹钟：工作日的 7-9 点和 17-19 点准时出现两个高峰，一年四季雷打不动。天气对他们有影响，但不大——下雨天少骑一点，但该上班还得上班。

casual 用户（休闲族）完全不一样。他们喜欢在周末的午后出门，遇到好天气蜂拥而出，下雨或太冷就几乎为零。他们的骑行量对温度的弹性远超 registered。

#### 天气说了算，但不是全部

温度和湿度是最重要的天气变量，但风和极端天气的影响相对有限。原因很朴素：DC 大部分日子天气都还行，真正让骑车人犹豫的不是"要不要带伞"，而是"外面冷不冷"。

#### 2012 年发生了什么

2012 年的租车量比 2011 年高了约 70%。共享单车在 DC 进入了快速普及期——这个趋势本身就是很强的预测信号。如果把 2011 年的模式不加调整地用在 2012 年，会系统性低估。

#### 小时是最强的维度

在所有特征中，`hour` 和 `hour_x_workingday` 始终排在重要度前三。一天 24 小时的节奏，是这座城市的呼吸频率。

### 8.5 复盘与后续方向

**这次做对了什么：**

1. EDA 阶段发现 casual 和 registered 行为差异 → 分头预测策略有数据支撑，不是拍脑袋的决定
2. `hour × workingday` 交互项捕获了"同一小时在不同日子的含义不同"这个关键信息
3. TimeSeriesSplit 保证了 CV 评估不会高估模型在真实场景下的表现
4. Stacking 集成了三种不同性质的模型，互补性不错

**可以改进的地方：**

1. 没有使用滞后特征（测试集无法计算），但可以用伪迭代预测或 day-of-week 均值做代理
2. weather=4 只有 1 条数据，模型对这个类别的预测完全不可靠——好在实际中也极少发生
3. 只用了树模型的 feature_importance 和 SHAP，没有做 permutation importance 的交叉验证
4. 2012 年整体增长 70% 的宏观趋势被 `year` 特征捕获了，但没有更精细的月内趋势建模

**如果再做一轮迭代（v2）：**
- 构造 weekend × hour 的二阶交互（替代 workingday × hour 的二元切分）
- 引入 smoothing / rolling features 作为趋势代理
- 尝试 Optuna 做更系统的超参搜索
- 对 weather=3/4 的样本做 oversampling 或加权处理

---

### 8.6 框架迁移

这套分析框架（时间序列分解 → 行为分群 → 分头建模 → 集成）可以迁移到任何"需要预测时序需求量"的场景：

- 外卖平台的订单量预测（午餐 vs 晚餐 vs 夜宵 = casual vs registered）
- 充电桩的使用率预测（通勤充电 vs 长途充电）
- 公共交通的客流预测

核心思路是不变的：**先理解数据里的行为模式，再设计模型去匹配它，而不是反过来让数据去迁就模型**。

---

以上就是 Bike Sharing Demand 分析的完整过程。感谢阅读。